# LEGO station: auto-label and train YOLO (Phase 3)

Runs on a **free Colab GPU**. Nothing here costs money or uses an API.

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save.

Then run the cells **top to bottom** (Shift+Enter, or Runtime → Run all). You will:

1. Upload `frames.zip` (made by `python vision/extract_frames.py` on your laptop).
2. Let Grounding DINO (open-source) draw boxes around the three parts on every frame.
3. Look at a sample of those boxes to check they make sense.
4. Train a small YOLO model on them (about 10–20 minutes).
5. Download `best.pt` and put it in `vision/models/` on your laptop.

Classes: `0 plate` (white 4×6), `1 pink_brick` (pink 1×6), `2 blue_slope` (light-blue 2×2).

In [ ]:
# 1. Check the GPU and install YOLO
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU! Runtime > Change runtime type > T4 GPU"
!pip -q install -U ultralytics
import torch; print("GPU available:", torch.cuda.is_available())

In [ ]:
# 2. Upload frames.zip from your laptop (vision/frames.zip)
import os, zipfile, glob
from pathlib import Path

WORK = Path("/content/lego")
WORK.mkdir(exist_ok=True)
if not glob.glob(str(WORK / "frames" / "*.jpg")):
    from google.colab import files
    print("Choose vision/frames.zip ...")
    uploaded = files.upload()
    for name in uploaded:
        zipfile.ZipFile(name).extractall(WORK)
FRAMES = sorted(glob.glob(str(WORK / "frames" / "*.jpg")))
print(len(FRAMES), "frames ready")

In [ ]:
# 3a. Labeling settings (you normally don't need to change these)
CLASSES = ["plate", "pink_brick", "blue_slope"]
PROMPT = "a white lego plate. a pink lego brick. a light blue lego brick."
BOX_THRESHOLD = 0.30      # lower = more boxes (and more mistakes)
TEXT_THRESHOLD = 0.25
DINO_MODEL = "IDEA-Research/grounding-dino-base"

In [ ]:
# 3b. Helper: turn Grounding DINO output into at most one box per part
import numpy as np, cv2

def word_to_class(text):
    t = text.lower()
    if "plate" in t: return 0
    if "pink" in t: return 1
    if "blue" in t: return 2
    return None

def color_to_class(img, box):
    # Fallback when the text label is unclear: look at the colour inside the box.
    x1, y1, x2, y2 = [int(v) for v in box]
    crop = img[max(0, y1):y2, max(0, x1):x2]
    if crop.size == 0: return None
    hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV).reshape(-1, 3)
    bright = hsv[hsv[:, 2] > 90]              # ignore the dark table
    if len(bright) < 20: return None
    h, s = np.median(bright[:, 0]), np.median(bright[:, 1])
    if s < 45: return 0                       # white / beige -> plate
    if 75 <= h <= 115: return 2               # blue-ish
    if h >= 140 or h <= 12: return 1          # pink-ish
    return None

def iou(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1]); x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / ua if ua else 0

def pick_boxes(img, boxes, scores, texts):
    """Each part exists only once, so keep the best box per class and drop obvious junk."""
    H, W = img.shape[:2]
    best = {}
    for box, score, text in zip(boxes, scores, texts):
        x1, y1, x2, y2 = box
        area = (x2 - x1) * (y2 - y1) / (W * H)
        if area > 0.6 or area < 0.0005:       # whole-table box, or a speck
            continue
        cls = word_to_class(text)
        col = color_to_class(img, box)
        if cls is None or (col is not None and col != cls and score < 0.45):
            cls = col                         # trust colour when the text is unsure
        if cls is None:
            continue
        if cls not in best or score > best[cls][1]:
            best[cls] = (list(map(float, box)), float(score))
    # A pink/blue box that is really the whole assembled plate -> drop it
    if 0 in best:
        for c in (1, 2):
            if c in best and iou(best[c][0], best[0][0]) > 0.7:
                del best[c]
    return {c: b for c, (b, s) in best.items()}

def to_yolo_lines(boxes_by_class, W, H):
    lines = []
    for c, (x1, y1, x2, y2) in sorted(boxes_by_class.items()):
        x1, x2 = max(0, x1), min(W, x2); y1, y2 = max(0, y1), min(H, y2)
        lines.append(f"{c} {(x1+x2)/2/W:.6f} {(y1+y2)/2/H:.6f} {(x2-x1)/W:.6f} {(y2-y1)/H:.6f}")
    return lines

In [ ]:
# 3c. Auto-label every frame with Grounding DINO (about 2-4 minutes on a T4)
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
from PIL import Image
from tqdm.auto import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
processor = AutoProcessor.from_pretrained(DINO_MODEL)
dino = AutoModelForZeroShotObjectDetection.from_pretrained(DINO_MODEL).to(device).eval()

def detect(pil):
    inputs = processor(images=pil, text=PROMPT, return_tensors="pt").to(device)
    with torch.no_grad():
        out = dino(**inputs)
    kw = dict(text_threshold=TEXT_THRESHOLD, target_sizes=[pil.size[::-1]])
    try:
        r = processor.post_process_grounded_object_detection(out, inputs.input_ids, threshold=BOX_THRESHOLD, **kw)[0]
    except TypeError:  # older transformers
        r = processor.post_process_grounded_object_detection(out, inputs.input_ids, box_threshold=BOX_THRESHOLD, **kw)[0]
    texts = r.get("text_labels", r.get("labels"))
    return r["boxes"].cpu().numpy(), r["scores"].cpu().numpy(), list(texts)

LABELS = WORK / "labels_all"
LABELS.mkdir(exist_ok=True)
counts = {c: 0 for c in CLASSES}
for f in tqdm(FRAMES):
    pil = Image.open(f).convert("RGB")
    img = cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)
    boxes, scores, texts = detect(pil)
    chosen = pick_boxes(img, boxes, scores, texts)
    for c in chosen: counts[CLASSES[c]] += 1
    (LABELS / (Path(f).stem + ".txt")).write_text("\n".join(to_yolo_lines(chosen, *pil.size)))
print("Frames with each part found:", counts, "out of", len(FRAMES))

## 4. Check the boxes

Look at the grid below. **White = plate, pink = pink brick, blue = blue slope.**
A few missed or slightly-off boxes are fine (hands block the parts a lot). If most boxes are
wrong, change `BOX_THRESHOLD` in cell 3a (try 0.25 or 0.35) and run 3c again.

In [ ]:
# 4. Show 24 random labeled frames
import random, matplotlib.pyplot as plt
COLORS = {0: (255, 255, 255), 1: (255, 105, 180), 2: (0, 191, 255)}

def draw(f):
    img = cv2.cvtColor(cv2.imread(f), cv2.COLOR_BGR2RGB); H, W = img.shape[:2]
    lab = LABELS / (Path(f).stem + ".txt")
    for line in lab.read_text().splitlines() if lab.exists() else []:
        c, x, y, w, h = line.split(); c = int(c)
        x, y, w, h = float(x)*W, float(y)*H, float(w)*W, float(h)*H
        cv2.rectangle(img, (int(x-w/2), int(y-h/2)), (int(x+w/2), int(y+h/2)), COLORS[c], max(2, W//200))
    return img

random.seed(0)
sample = random.sample(FRAMES, min(24, len(FRAMES)))
fig, axes = plt.subplots(4, 6, figsize=(24, 16))
for ax, f in zip(axes.flat, sample):
    ax.imshow(draw(f)); ax.set_title(Path(f).stem, fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# 5. Build the YOLO dataset (85% train / 15% check)
import shutil, yaml
DS = WORK / "dataset"
shutil.rmtree(DS, ignore_errors=True)
random.seed(42)
shuffled = FRAMES[:]; random.shuffle(shuffled)
n_val = max(1, int(len(shuffled) * 0.15))
for split, files_ in (("val", shuffled[:n_val]), ("train", shuffled[n_val:])):
    (DS / "images" / split).mkdir(parents=True); (DS / "labels" / split).mkdir(parents=True)
    for f in files_:
        shutil.copy(f, DS / "images" / split)
        lab = LABELS / (Path(f).stem + ".txt")
        shutil.copy(lab, DS / "labels" / split) if lab.exists() else (DS / "labels" / split / lab.name).write_text("")
(DS / "data.yaml").write_text(yaml.safe_dump({"path": str(DS), "train": "images/train", "val": "images/val",
                                             "names": {i: n for i, n in enumerate(CLASSES)}}))
print("train:", len(shuffled) - n_val, " val:", n_val)

In [ ]:
# 6. Train YOLO (about 10-20 minutes on a T4)
from ultralytics import YOLO
try:
    model = YOLO("yolo26n.pt")      # newest small model
except Exception:
    model = YOLO("yolo11n.pt")      # fallback
results = model.train(
    data=str(DS / "data.yaml"), epochs=80, patience=25, imgsz=640, batch=16,
    project=str(WORK / "runs"), name="lego", exist_ok=True,
    # Overhead camera: any rotation/flip is realistic. Strong colour/brightness changes = robust to lighting.
    degrees=180, flipud=0.5, fliplr=0.5, hsv_h=0.02, hsv_s=0.6, hsv_v=0.5, scale=0.4,
)
BEST = WORK / "runs" / "lego" / "weights" / "best.pt"
print("Saved:", BEST)

In [ ]:
# 7. How good is it? (mAP50 near 1.0 = very good, above 0.8 = fine for the demo)
best = YOLO(str(BEST))
m = best.val(data=str(DS / "data.yaml"), verbose=False)
print(f"mAP50: {m.box.map50:.3f}   mAP50-95: {m.box.map:.3f}")
for i, n in enumerate(CLASSES):
    print(f"  {n:11s} mAP50 = {m.box.ap50[i]:.3f}" if i < len(m.box.ap50) else f"  {n}: no data")

val_imgs = sorted(glob.glob(str(DS / "images" / "val" / "*.jpg")))[:6]
fig, axes = plt.subplots(1, len(val_imgs), figsize=(4 * len(val_imgs), 5))
for ax, f in zip(np.atleast_1d(axes), val_imgs):
    ax.imshow(best.predict(f, conf=0.4, verbose=False)[0].plot()[:, :, ::-1]); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# 8. Download the trained model -> put it in vision/models/best.pt on your laptop
shutil.copy(BEST, "/content/best.pt")
shutil.make_archive("/content/lego_dataset", "zip", DS)   # labeled data, handy for retraining later
from google.colab import files
files.download("/content/best.pt")